## 1. 安装ragas

```shell
# 方式-1
pip install ragas

# 方式-2
git clone https://github.com/explodinggradients/ragas.git 
cd ragas 
pip install -e .
```

## 2. 生成综合测试集

In [2]:
# 加载文件

from langchain_community.document_loaders import DirectoryLoader

In [3]:
loader = DirectoryLoader("files/", 
                         show_progress=True, 
                         use_multithreading=True,
                         max_concurrency=6)

In [4]:
documents = loader.load()

  0%|          | 0/2 [00:00<?, ?it/s]This function will be deprecated in a future release and `unstructured` will simply use the DEFAULT_MODEL from `unstructured_inference.model.base` to set default model name
This function will be deprecated in a future release and `unstructured` will simply use the DEFAULT_MODEL from `unstructured_inference.model.base` to set default model name
Some weights of the model checkpoint at microsoft/table-transformer-structure-recognition were not used when initializing TableTransformerForObjectDetection: ['model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing TableTransformerForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model fr

In [5]:
for document in documents:
    print(document)
    break

[Document(page_content='1. 故宫，⼜称紫禁城，位于中国北京中⼼，是明朝和清朝两代皇帝的皇宫。这座宏伟的宫殿群建筑于1420年完 成，拥有超过600年的历史，是世界上现存规模最⼤、保存最为完整的⽊结构古建筑之⼀。\n\n2. 故宫占地⾯积达72万平⽅⽶，共有宫殿⼤殿、殿堂、楼阁、以及众多院落9000余间，是⼀座规模宏⼤的建筑 杰作。整个宫殿⾃南向北依次是太和⻔、太和殿、中和殿、保和殿，每⼀处都有其独特的历史和⽂化价值。\n\n3. 故宫不仅是中国历史上的皇宫，还是⼀个丰富的⽂化艺术宝库。它收藏了⼤量的艺术品，包括古代中国的绘 画、书法、陶瓷、⽟器等珍贵⽂物。\n\n4. 故宫的建筑设计和装饰展现了中国古代建筑艺术的精髓。宫墙周围是宽阔的护城河，四⻆有巍峨的⻆楼，不 仅增强了宫殿的防御功能，也体现了古代建筑的美学追求。\n\n5. 故宫的中轴线是北京城的中轴线，从南⻔即午⻔开始，通过太和⻔、太和殿，直⾄北⻔神武⻔，形成了⼀条 严格的对称轴线。这不仅反映了古代中国的宇宙观，也体现了皇权⾄上的思想。\n\n6. 在故宫的北部，有⼀个宁静美丽的皇家园林——御花园。这⾥种植着各种珍稀植物，还有精美的亭台楼阁， 是皇族成员休闲娱乐的地⽅。\n\n7. 故宫的每⼀处建筑都有其特定的功能和象征意义。例如，太和殿是皇帝举⾏⼤典和接⻅外国使节的地⽅，象 征着皇权的⾄⾼⽆上。\n\n8. 故宫还拥有⼀批珍贵的钟表收藏，这些来⾃世界各地的时钟和钟表不仅⼯艺精美，⽽且技术先进，展示了清 代中国与世界其他地区的交流与融合。\n\n9. 故宫不只是⼀座宫殿，它还是研究中国历史、⽂化、艺术和建筑的重要场所。每年吸引着来⾃世界各地的游 客和学者，是中国最受欢迎的旅游景点之⼀。\n\n10. 今天的故宫已经转变成为中国故宫博物院，致⼒于⽂物的保护和研究⼯作，让这座古⽼的皇宫以⼀种新的⽅ 式继续其⽂化传承，让更多⼈了解和研究中国丰富的历史和⽂化。', metadata={'source': 'files/故宫介绍.pdf'})]


In [7]:
# 数据生成

from ragas.testset.generator import TestsetGenerator
from ragas.testset.evolutions import simple, reasoning, multi_context
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [8]:
generator_llm = ChatOpenAI(model="gpt-3.5-turbo")

critic_llm = ChatOpenAI(model="gpt-3.5-turbo-1106")

embeddings = OpenAIEmbeddings()

In [9]:
generator = TestsetGenerator.from_langchain(
    generator_llm,
    critic_llm,
    embeddings
)

In [10]:
testset = generator.generate_with_langchain_docs(
    documents[0],
    test_size=2,
    distributions={simple:0.5,
                   reasoning: 0.25,
                   multi_context: 0.25}
)

Filename and doc_id are the same for all nodes.               
Generating: 100%|██████████| 2/2 [00:29<00:00, 14.80s/it]


In [11]:
testset

TestDataset(test_data=[DataRow(question='故宫如何转变成为中国故宫博物院，并致力于什么样的工作？', contexts=['。\n\n7. 故宫的每⼀处建筑都有其特定的功能和象征意义。例如，太和殿是皇帝举⾏⼤典和接⻅外国使节的地⽅，象 征着皇权的⾄⾼⽆上。\n\n8. 故宫还拥有⼀批珍贵的钟表收藏，这些来⾃世界各地的时钟和钟表不仅⼯艺精美，⽽且技术先进，展示了清 代中国与世界其他地区的交流与融合。\n\n9. 故宫不只是⼀座宫殿，它还是研究中国历史、⽂化、艺术和建筑的重要场所。每年吸引着来⾃世界各地的游 客和学者，是中国最受欢迎的旅游景点之⼀。\n\n10. 今天的故宫已经转变成为中国故宫博物院，致⼒于⽂物的保护和研究⼯作，让这座古⽼的皇宫以⼀种新的⽅ 式继续其⽂化传承，让更多⼈了解和研究中国丰富的历史和⽂化。'], ground_truth='今天的故宫已经转变成为中国故宫博物院，致力于文物的保护和研究工作。', evolution_type='simple', metadata=[{'source': 'files/故宫介绍.pdf'}]), DataRow(question='为何故宫建筑注重功能和象征意义？', contexts=['。\n\n7. 故宫的每⼀处建筑都有其特定的功能和象征意义。例如，太和殿是皇帝举⾏⼤典和接⻅外国使节的地⽅，象 征着皇权的⾄⾼⽆上。\n\n8. 故宫还拥有⼀批珍贵的钟表收藏，这些来⾃世界各地的时钟和钟表不仅⼯艺精美，⽽且技术先进，展示了清 代中国与世界其他地区的交流与融合。\n\n9. 故宫不只是⼀座宫殿，它还是研究中国历史、⽂化、艺术和建筑的重要场所。每年吸引着来⾃世界各地的游 客和学者，是中国最受欢迎的旅游景点之⼀。\n\n10. 今天的故宫已经转变成为中国故宫博物院，致⼒于⽂物的保护和研究⼯作，让这座古⽼的皇宫以⼀种新的⽅ 式继续其⽂化传承，让更多⼈了解和研究中国丰富的历史和⽂化。'], ground_truth='故宫的每一处建筑都有其特定的功能和象征意义，比如太和殿是皇帝举行大典和接待外国使节的地方，象征着皇权的至高无上。因此，故宫建筑注重功能和象征意义是为了体现皇权和展示特定用途。', evolution_type='reasoning', metadata=[{'sourc

In [12]:
testset.to_pandas()

,question,contexts,ground_truth,evolution_type,metadata,episode_done
0,故宫如何转变成为中国故宫博物院，并致力于什么样的工作？,[。\n\n7. 故宫的每⼀处建筑都有其特定的功能和象征意义。例如，太和殿是皇帝举⾏⼤典和接...,今天的故宫已经转变成为中国故宫博物院，致力于文物的保护和研究工作。,simple,[{'source': 'files/故宫介绍.pdf'}],True
1,为何故宫建筑注重功能和象征意义？,[。\n\n7. 故宫的每⼀处建筑都有其特定的功能和象征意义。例如，太和殿是皇帝举⾏⼤典和接...,故宫的每一处建筑都有其特定的功能和象征意义，比如太和殿是皇帝举行大典和接待外国使节的地方，象...,reasoning,[{'source': 'files/故宫介绍.pdf'}],True


## 3. 使用自己的测试集进行评估

In [13]:
from datasets import load_dataset

In [14]:
amnesty_qa = load_dataset("explodinggradients/amnesty_qa", 
                          "english_v2")

Generating eval split: 20 examples [00:00, 1484.37 examples/s]


In [15]:
amnesty_qa

DatasetDict({
    eval: Dataset({
        features: ['question', 'ground_truth', 'answer', 'contexts'],
        num_rows: 20
    })
})

In [16]:
# 评估指标
from ragas.metrics import (
    answer_relevancy,
    faithfulness,
    context_recall,
    context_precision,
)

In [19]:
# 评估

from ragas import evaluate

result = evaluate(
    amnesty_qa["eval"],
    metrics=[
        context_precision,
        faithfulness,
        answer_relevancy,
        context_recall,
    ]
)

Evaluating: 100%|██████████| 80/80 [00:27<00:00,  2.87it/s]


In [20]:
df = result.to_pandas()

In [21]:
df.head()

,question,ground_truth,answer,contexts,context_precision,faithfulness,answer_relevancy,context_recall
0,What are the global implications of the USA Su...,The global implications of the USA Supreme Cou...,The global implications of the USA Supreme Cou...,"[- In 2022, the USA Supreme Court handed down ...",1.0,1.000000,0.988014,1.0
1,Which companies are the main contributors to G...,"According to the Carbon Majors database, the m...","According to the Carbon Majors database, the m...","[- Fossil fuel companies, whether state or pri...",1.0,1.000000,0.965893,1.0
2,Which private companies in the Americas are th...,The largest private companies in the Americas ...,"According to the Carbon Majors database, the l...",[The private companies responsible for the mos...,1.0,0.300000,0.988343,1.0
3,What action did Amnesty International urge its...,Amnesty International urged its supporters to ...,Amnesty International urged its supporters to ...,[Amnesty International called on its vast netw...,1.0,0.600000,0.932876,1.0
4,What are the recommendations made by Amnesty I...,The recommendations made by Amnesty Internatio...,Amnesty International made several recommendat...,[Amnesty International recommends that the Spe...,1.0,0.090909,0.993758,1.0


## 4. 核心概念

Ragas旨在通过提供一系列工具和技术，使开发者能够在RAG应用中有效利用持续学习。主要功能包括：

1. **合成测试数据集生成**：自动生成多样化测试数据，全面评估应用表现。
2. **基于LLM的评估指标**：利用LLM辅助评估，客观衡量应用性能。
3. **生产质量监控**：使用成本较低的模型实时监控应用质量，提供可操作的见解。
4. **应用迭代改进**：基于实时反馈和深入分析，持续优化产品。

Ragas的目标是通过持续学习，提高RAG应用的质量和用户满意度。

### 4.1 指标驱动开发(Metrics-Driven Development, MDD)

MDD是一种依赖数据做出明智决策的产品开发方法。这种方法涉及随时间持续监测关键指标，为应用程序的性能提供宝贵见解。


- **关键功能**

1. **评估**：能够以指标辅助的方式评估LLM应用程序并进行实验，确保高度的可靠性和可复制性。
   
2. **监控**：从生产数据点中获得宝贵且可操作的见解，促进LLM应用的质量持续改进。


### 4.2 指标（Metrics）

#### 4.2.1 忠诚度（Faithfulness）

- **定义**：测量生成答案与给定上下文的事实一致性。
- **计算方法**：根据答案和检索到的上下文，将答案的分数缩放到（0,1）范围，分数越高表示越好。
- **评估过程**：首先识别生成答案中的声明集合，然后检查这些声明是否能从给定上下文中推断出来。忠实度分数由能从上下文中推断出的声明数量除以生成答案中声明的总数得出。

$\text{忠诚度分数} = {|\text{从上下文中推断出的声明数量}| \over |\text{生成答案中声明的总数}|}$

  
   
**示例分析：**

**问题：** 爱因斯坦何时何地出生？  
**上下文：** 阿尔伯特·爱因斯坦（1879年3月14日出生）是一位德国出生的理论物理学家，被广泛认为是有史以来最伟大和最有影响力的科学家之一。  

**高忠实度答案分析：**  
**答案：** 爱因斯坦于1879年3月14日在德国出生。   
**忠实度评估：** 该答案与提供的上下文完全一致，准确反映了爱因斯坦的出生时间和地点，因此具有高忠实度。  

**低忠实度答案分析：**  
**答案：** 爱因斯坦于1879年3月20日在德国出生。   
**忠实度评估：** 虽然该答案正确指出了爱因斯坦的出生地，但出生日期与上下文给出的信息不符，因此具有低忠实度。 

我们来看看如何使用低忠实度答案来计算忠实度：

- **步骤1**：将生成的答案分解成单独的声明。
  - 声明：
    - 声明1：爱因斯坦出生在德国
    - 声明2：爱因斯坦出生于1879年3月20日

- **步骤2**：对于每个生成的声明，验证它是否可以从给定的上下文中推断出来。
  - 声明1：可以（Yes）
  - 声明2：不可以（No）

- **步骤3**：使用上面描述的公式来计算忠实度。
  - $忠实度 = \frac{1}{2} = 0.5$

这里，通过计算与给定上下文一致的声明的比例来确定忠实度分数，从而得到一个介于0到1之间的值，反映了答案与上下文的一致程度。在这个例子中，由于两个声明中只有一个与上下文一致，忠实度分数为0.5。

In [22]:
from datasets import Dataset
from ragas.metrics import faithfulness
from ragas import evaluate

data_samples = {
    'question': ['中国第一次举办奥运会是什么时候？', '哪个城市是中国历史上第一个举办亚运会的？'],
    'answer': ['中国第一次举办奥运会是在2008年', '中国历史上第一个举办亚运会的城市是北京'],
    'contexts': [['2008年夏季奥林匹克运动会，也称为北京2008奥运会，是一项国际多项体育赛事，于2008年8月8日至8月24日在中国北京举行。'], 
    ['第11届亚洲运动会，也称为1990年北京亚运会，是一项在中国北京举行的国际综合体育赛事。']],
}

dataset = Dataset.from_dict(data_samples)

score = evaluate(dataset, metrics=[faithfulness])

score.to_pandas()

Evaluating: 100%|██████████| 2/2 [00:02<00:00,  1.32s/it]


,question,answer,contexts,faithfulness
0,中国第一次举办奥运会是什么时候？,中国第一次举办奥运会是在2008年,[2008年夏季奥林匹克运动会，也称为北京2008奥运会，是一项国际多项体育赛事，于2008...,1.0
1,哪个城市是中国历史上第一个举办亚运会的？,中国历史上第一个举办亚运会的城市是北京,[第11届亚洲运动会，也称为1990年北京亚运会，是一项在中国北京举行的国际综合体育赛事。],1.0


#### 4.2.2 答案相关性（Answer Relevance）

评估指标“答案相关性”关注于评估生成答案与给定提示的相关性有多强。  
较低的分数被赋予那些不完整或包含冗余信息的答案，而较高的分数表明答案相关性更好。  
这个指标是利用问题（question）、上下文（context）和答案（answer）来计算的。

**定义：** 为原始问题到一系列基于答案生成（反向工程）的人工问题的平均余弦相似度。它的计算公式如下：

$\text{答案相关性} = \frac{1}{N} \sum_{i=1}^{N} \cos(E_{gi}, E_{o})$

其中：

- $E_{gi}$ 是生成问题 i 的嵌入。
- $E_{o}$ 是原始问题的嵌入。
- $N$ 是生成问题的数量，默认为3。

请注意，尽管在实践中这个分数通常会介于0和1之间，但这并不是数学上保证的，因为余弦相似度的范围是从-1到1。


**答案相关性详解:**  

当一个答案直接且恰当地回应了原始问题时，它被认为是相关的。重要的是，答案相关性的评估并不考虑事实性，而是对答案不完整或包含冗余信息的情况进行惩罚。为了计算这个分数，会提示LLM（大型语言模型）根据生成的答案多次生成适当的问题，然后测量这些生成问题与原始问题之间的平均余弦相似度。核心思想是，如果生成的答案准确回答了初始问题，那么LLM应该能够生成与原始问题对齐的问题。

**示例:**  
- 问题：法国在哪里，它的首都是什么？  
- 低相关性答案：法国在西欧。  
- 高相关性答案：法国在西欧，巴黎是它的首都。

要计算答案与给定问题的相关性，我们遵循以下两个步骤：

**步骤1：**  
利用大型语言模型（LLM）从生成的答案反向工程出多个问题的变体。例如，对于第一个答案，LLM可能会生成以下可能的问题：

问题1：法国位于欧洲的哪个部分？  
问题2：法国在欧洲的地理位置是什么？  
问题3：您能确定法国位于欧洲的哪个区域吗？  

**步骤2：**  
计算生成问题与实际问题之间的平均余弦相似度。  

核心概念是，如果答案正确地解决了问题，那么就有很高的可能性仅从答案中就能重建出原始问题。这意味着，答案的信息足够充分，使得可以通过它生成与原始问题密切相关的其他问题，从而通过平均余弦相似度的高低来衡量答案的相关性。

In [23]:
from datasets import Dataset
from ragas.metrics import answer_relevancy
from ragas import evaluate

data_samples = {
    'question': ['中国第一次举办奥运会是什么时候？', '哪个城市是中国历史上第一个举办亚运会的？'],
    'answer': ['中国第一次举办奥运会是在2008年', '中国历史上第一个举办亚运会的城市是北京'],
    'contexts': [['2008年夏季奥林匹克运动会，也称为北京2008奥运会，是一项国际多项体育赛事，于2008年8月8日至8月24日在中国北京举行。'], 
    ['第11届亚洲运动会，也称为1990年北京亚运会，是一项在中国北京举行的国际综合体育赛事。']],
}

dataset = Dataset.from_dict(data_samples)

score = evaluate(dataset, metrics=[answer_relevancy])

score.to_pandas()

Evaluating: 100%|██████████| 2/2 [00:03<00:00,  1.62s/it]


,question,answer,contexts,answer_relevancy
0,中国第一次举办奥运会是什么时候？,中国第一次举办奥运会是在2008年,[2008年夏季奥林匹克运动会，也称为北京2008奥运会，是一项国际多项体育赛事，于2008...,0.984297
1,哪个城市是中国历史上第一个举办亚运会的？,中国历史上第一个举办亚运会的城市是北京,[第11届亚洲运动会，也称为1990年北京亚运会，是一项在中国北京举行的国际综合体育赛事。],0.976234


#### 4.2.3 上下文精确度（Context Precision）

上下文精确度是一种衡量所有真实相关项是否被排在上下文中较高位置的指标。理想情况下，所有相关的内容块都应出现在最顶端的排名中。  

这个指标是利用问题（question）、真实答案（ground_truth）和上下文（contexts）来计算的，分值范围在0到1之间，分数越高表示精确度越好。

上下文精确度的计算公式如下：

$\text{Context Precision@K} = \frac{\sum_{k=1}^{K} (\text{Precision@k} \times v_k)}{\text{Total number of relevant items in the top } K \text{ results}}$

其中Precision@k的计算方式为：

$\text{Precision@k} = \frac{\text{true positives@k}}{\text{true positives@k} + \text{false positives@k}}$

这里的 $K$ 是上下文中块的总数，而 $v_k$ 是在排名 $k$ 处的相关性指标，取值为0或1。

该指标对于评估信息检索系统中，针对给定查询返回的上下文信息的精确性是非常有用的，尤其是在需要确定系统是否能够有效地识别和排列与查询最相关的信息块时。通过计算Precision@k，我们能够得知在前k个结果中正确相关信息的比例，而上下文精确度则是所有这些比例的加权平均。


**示例**

问题：法国在哪里，它的首都是什么？  
真实答案：法国位于西欧，首都是巴黎。

**高上下文精确度**：  
- “法国位于西欧，拥有中世纪城市、阿尔卑斯山村庄和地中海海滩。巴黎，其首都，以其时尚屋、古典艺术博物馆（包括卢浮宫）和像艾菲尔铁塔这样的纪念碑而闻名。”
- “这个国家也以其葡萄酒和精致菜肴而著称。拉斯科的古代洞穴画、里昂的罗马剧场和广阔的凡尔赛宫证明了其丰富的历史。”

**低上下文精确度**：  
- “这个国家也因其葡萄酒和精致的菜肴而闻名。拉斯科的古代洞穴画，里昂的罗马剧场和”
- “法国位于西欧，拥有中世纪城市、阿尔卑斯山村庄和地中海海滩。巴黎，其首都，以其时尚屋、古典艺术博物馆（包括卢浮宫）和像艾菲尔铁塔这样的纪念碑而闻名。”

在这个示例中，高上下文精确度的文本直接且全面地回答了问题，而低上下文精确度的文本虽然提及了一些相关信息，但是未能全面回答问题，且第二段话被突然截断，显示了不完整性。这说明了在评估上下文精确度时，我们不仅要考虑信息的相关性，还要考虑信息的完整性和结构性。  


**上下文精确度计算**:

如何使用低上下文精确度示例来计算上下文精确度：

- **步骤1**：对于检索到的上下文中的每个块，检查它是否与给定问题的真实答案相关。
  
- **步骤2**：为上下文中的每个块计算precision@k。
  - Precision@1 = 0 / 1 = 0  
  - Precision@2 = 1 / 2 = 0.5  
   
- **步骤3**：计算precision@k的平均值，以得出最终的上下文精确度分数。
  
$\text{上下文精确度} = \frac{(0 + 0.5)}{1} = 0.5$

这里显示的是一个具体的计算例子，其中precision@1为0，因为第一个块与问题不相关；而precision@2为0.5，因为在前两个块中，有一个与问题相关。这样计算出的上下文精确度为0.5。

In [25]:
from datasets import Dataset
from ragas.metrics import context_precision
from ragas import evaluate

data_samples = {
    'question': ['中国第一次举办奥运会是什么时候？', '哪个城市赢得了最多的中超联赛冠军？'],
    'answer': ['中国第一次举办奥运会是在2008年', '广州恒大足球俱乐部赢得了最多的中超联赛冠军'],
    'contexts': [['2008年夏季奥林匹克运动会，也称为北京2008奥运会，是一项国际多项体育赛事，于2008年8月8日至8月24日在中国北京举行。'], 
    ['广州恒大足球俱乐部...广州。','恒大队在...中国足球协会超级联赛']],
    'ground_truth': ['2008年夏季奥林匹克运动会于2008年8月8日在北京开幕', '广州恒大足球俱乐部赢得了八次中超联赛冠军']
}


dataset = Dataset.from_dict(data_samples)

score = evaluate(dataset, metrics=[context_precision])

score.to_pandas()

Evaluating: 100%|██████████| 2/2 [00:03<00:00,  1.68s/it]


,question,answer,contexts,ground_truth,context_precision
0,中国第一次举办奥运会是什么时候？,中国第一次举办奥运会是在2008年,[2008年夏季奥林匹克运动会，也称为北京2008奥运会，是一项国际多项体育赛事，于2008...,2008年夏季奥林匹克运动会于2008年8月8日在北京开幕,1.0
1,哪个城市赢得了最多的中超联赛冠军？,广州恒大足球俱乐部赢得了最多的中超联赛冠军,"[广州恒大足球俱乐部...广州。, 恒大队在...中国足球协会超级联赛]",广州恒大足球俱乐部赢得了八次中超联赛冠军,1.0


#### 4.2.4 上下文召回率（Context Recall）

上下文召回率衡量检索到的上下文与作为真实答案之间的一致性。它是基于真实答案（ground truth）和检索到的上下文（retrieved context）来计算的，值在0到1之间，数值越高表明性能越好。

为了从真实答案中估计上下文召回率，需要分析真实答案中的每个句子是否能归因于检索到的上下文。在理想情况下，真实答案中的所有句子都应该能归因于检索到的上下文。

计算上下文召回率的公式如下：

$\text{上下文召回率} = \frac{|能归因于上下文的GT句子|}{|GT中的句子数量|}$


**示例：**  

**问题：** 法国在哪里，它的首都是什么？  
**真实答案：** 法国在西欧，首都是巴黎。  

**高上下文召回率：** 法国位于西欧，包括中世纪城市、阿尔卑斯山村庄和地中海海滩。巴黎，其首都，以其时尚屋、包括卢浮宫在内的古典艺术博物馆和象征性的艾菲尔铁塔而闻名。  

**低上下文召回率：** 法国位于西欧，包括中世纪城市、阿尔卑斯山村庄和地中海海滩。这个国家也因其葡萄酒和精致菜肴而知名。拉斯科的古代洞穴画、里昂的罗马剧场和广阔的凡尔赛宫证明了其丰富的历史。  

这个指标有助于评估在回答问题时，检索到的上下文信息是否完整覆盖了真实答案的所有点。  



> 让我们来看一下如何使用低上下文召回率示例来计算上下文召回率：

- **步骤1**：将真实答案分解成单独的陈述。
  - 陈述：
    - 陈述1：“法国位于西欧。”
    - 陈述2：“它的首都是巴黎。”

- **步骤2**：对于每个真实答案的陈述，验证它是否可以归因于检索到的上下文。
  - 陈述1：是的（Yes）
  - 陈述2：不是（No）

- **步骤3**：使用上述公式来计算上下文召回率。
  - 上下文召回率 = $\frac{1}{2}$ = 0.5

这个计算过程展示了在给定示例中，真实答案的一半（即一个陈述）可以直接归因于检索到的上下文，因此上下文召回率是0.5。

In [5]:
from datasets import Dataset
from ragas.metrics import context_recall
from ragas import evaluate


data_samples = {
    'question': ['中国第一次举办奥运会是什么时候？', '哪个城市赢得了最多的中超联赛冠军？'],
    'answer': ['中国第一次举办奥运会是在2008年', '广州恒大足球俱乐部赢得了最多的中超联赛冠军'],
    'contexts': [['2008年夏季奥林匹克运动会，也称为北京2008奥运会，是一项国际多项体育赛事，于2008年8月8日至8月24日在中国北京举行。'], 
    ['广州恒大足球俱乐部...广州。','恒大队在...中国足球协会超级联赛']],
    'ground_truth': ['2008年夏季奥林匹克运动会于2008年8月8日在北京开幕', '广州恒大足球俱乐部赢得了八次中超联赛冠军']
}


dataset = Dataset.from_dict(data_samples)

score = evaluate(dataset, metrics=[context_recall])

score.to_pandas()

Evaluating: 100%|██████████| 2/2 [00:02<00:00,  1.36s/it]


,question,answer,contexts,ground_truth,context_recall
0,中国第一次举办奥运会是什么时候？,中国第一次举办奥运会是在2008年,[2008年夏季奥林匹克运动会，也称为北京2008奥运会，是一项国际多项体育赛事，于2008...,2008年夏季奥林匹克运动会于2008年8月8日在北京开幕,1.0
1,哪个城市赢得了最多的中超联赛冠军？,广州恒大足球俱乐部赢得了最多的中超联赛冠军,"[广州恒大足球俱乐部...广州。, 恒大队在...中国足球协会超级联赛]",广州恒大足球俱乐部赢得了八次中超联赛冠军,1.0


#### 4.2.5 上下文实体召回率（Context Entities Recall）

这个指标衡量了检索到的上下文中实体的召回情况，基于真实答案（ground_truths）和上下文（contexts）中的实体数量，相对于仅在真实答案中的实体数量。  

简单来说，它测量了从真实答案召回的实体占比。这个指标对于基于事实的用例（如旅游咨询台、历史问答等）非常有用，因为它能够评估实体检索机制的效果，通过与真实答案中出现的实体进行比较，特别是在实体重要的情况下，需要上下文中包含这些实体。

为了计算这个指标，我们使用两组集合，$GE$ 和 $CE$ ，分别代表真实答案中的实体集合和上下文中的实体集合。然后我们取这些集合的交集中元素的数量，并将其除以 $GE$ 集合中元素的数量，公式如下：

$\text{上下文实体召回率} = \frac{|CE \cap GE|}{|GE|}$

**示例：**  

**真实答案：** 泰姬陵是位于印度阿格拉亚穆纳河右岸的象牙白大理石陵墓。它由莫卧儿帝国皇帝沙贾汗于1631年下令建造，以纪念他最爱的妻子，姆塔兹·玛哈尔。

**高实体召回上下文：** 泰姬陵是爱情和建筑奇迹的象征，位于印度阿格拉。它由莫卧儿帝国皇帝沙贾汗建造，纪念他的爱妻姆塔兹·玛哈尔。这座结构以其错综复杂的大理石工艺和周围美丽的花园而著称。

**低实体召回上下文：** 泰姬陵是印度的标志性纪念碑。它是联合国教科文组织的世界遗产，每年吸引数百万游客。复杂的雕刻和惊人的建筑使它成为必游之地。

通过这个指标，可以量化检索到的上下文中的实体与问题的真实答案的匹配程度。  


> 让我们考虑上述给出的真实答案和上下文。

- **步骤1**：找出真实答案中的实体。
  - 真实答案中的实体（GE）- ['泰姬陵', '亚穆纳河', '阿格拉', '1631', '沙贾汗', '姆塔兹·玛哈尔']

- **步骤2**：找出上下文中的实体。
  - 上下文中的实体（CE1）- ['泰姬陵', '阿格拉', '沙贾汗', '姆塔兹·玛哈尔', '印度']
  - 上下文中的实体（CE2）- ['泰姬陵', '联合国教科文组织', '印度']

- **步骤3**：使用上述公式来计算实体-召回。
  - 上下文实体召回率1 = |CE1 ∩ GE| / |GE| = 4/6 = 0.666
  - 上下文实体召回率2 = |CE2 ∩ GE| / |GE| = 1/6 = 0.166

这说明第一个上下文在给定的真实答案基础上具有较高的实体召回率，因为它覆盖了更多的真实答案中的实体。如果这两个上下文是通过两个不同的检索机制在同一组文档上检索到的，我们可以认为第一个机制在实体重要的使用场景中比第二个机制表现更好。

In [6]:
from datasets import Dataset
from ragas.metrics import context_entity_recall
from ragas import evaluate

data_samples = {
    'question': ['中国第一次举办奥运会是什么时候？', '哪个城市赢得了最多的中超联赛冠军？'],
    'answer': ['中国第一次举办奥运会是在2008年', '广州恒大足球俱乐部赢得了最多的中超联赛冠军'],
    'contexts': [['2008年夏季奥林匹克运动会，也称为北京2008奥运会，是一项国际多项体育赛事，于2008年8月8日至8月24日在中国北京举行。'], 
    ['广州恒大足球俱乐部...广州。','恒大队在...中国足球协会超级联赛']],
    'ground_truth': ['2008年夏季奥林匹克运动会于2008年8月8日在北京开幕', '广州恒大足球俱乐部赢得了八次中超联赛冠军']
}

dataset = Dataset.from_dict(data_samples)

score = evaluate(dataset, metrics=[context_entity_recall])

score.to_pandas()

Evaluating: 100%|██████████| 2/2 [00:03<00:00,  1.55s/it]


,question,answer,contexts,ground_truth,context_entity_recall
0,中国第一次举办奥运会是什么时候？,中国第一次举办奥运会是在2008年,[2008年夏季奥林匹克运动会，也称为北京2008奥运会，是一项国际多项体育赛事，于2008...,2008年夏季奥林匹克运动会于2008年8月8日在北京开幕,0.333333
1,哪个城市赢得了最多的中超联赛冠军？,广州恒大足球俱乐部赢得了最多的中超联赛冠军,"[广州恒大足球俱乐部...广州。, 恒大队在...中国足球协会超级联赛]",广州恒大足球俱乐部赢得了八次中超联赛冠军,0.500000


#### 4.2.6 答案语义相似性（Answer semantic similarity）

答案语义相似性的概念涉及对生成答案和真实答案之间语义相似度的评估。这种评估是基于真实答案（ground truth）和答案（answer），数值范围在0到1之间。分数越高，表示生成答案与真实答案之间的对齐越好。

衡量答案之间的语义相似性可以提供对生成响应质量的有价值的洞见。这种评估利用跨编码器模型来计算语义相似性分数。

**示例：**

**真实答案：** 阿尔伯特·爱因斯坦的相对论彻底改变了我们对宇宙的理解。

**高相似性答案：** 爱因斯坦的开创性相对论改变了我们对宇宙的理解。

**低相似性答案：** 艾萨克·牛顿的运动定律对古典物理学产生了巨大影响。

通过这个示例，我们可以看到答案语义相似性如何区分与真实答案相关性高低的生成答案。高相似性答案与真实答案紧密相关，而低相似性答案虽然可能在科学的范畴内，但与特定的真实答案关联不大。

In [7]:
from datasets import Dataset
from ragas.metrics import answer_similarity
from ragas import evaluate

data_samples = {
    'question': ['中国的首都是哪里？', '世界上最长的河流是哪一条？'],
    'answer': ['中国的首都是北京', '世界上最长的河流是尼罗河'],
    'ground_truth': ['北京是中华人民共和国的首都', '世界上最长的河流是尼罗河，它流经多个非洲国家']
}


dataset = Dataset.from_dict(data_samples)

score = evaluate(dataset, metrics=[answer_similarity])

score.to_pandas()

Evaluating: 100%|██████████| 2/2 [00:01<00:00,  1.47it/s]


,question,answer,ground_truth,answer_similarity
0,中国的首都是哪里？,中国的首都是北京,北京是中华人民共和国的首都,0.942584
1,世界上最长的河流是哪一条？,世界上最长的河流是尼罗河,世界上最长的河流是尼罗河，它流经多个非洲国家,0.949156


#### 4.2.7 答案正确性（Answer Correctness）

答案正确性的评估涉及衡量生成答案相对于真实答案的准确性。这种评价依赖于真实答案（ground truth）和答案（answer），得分范围从0到1。更高的分数表示生成答案与真实答案之间对齐更紧密，也就意味着更好的正确性。

答案正确性包含两个关键方面：生成答案与真实答案之间的语义相似性以及事实相似性。这些方面通过加权方案结合起来，以制定答案正确性分数。用户还可以选择使用“阈值”将结果分数转换为二元值（即正确或错误）。

**示例:**  

**真实答案：** 爱因斯坦于1879年在德国出生。
**高答案正确性：** 在1879年，爱因斯坦出生在德国。
**低答案正确性：** 爱因斯坦于1879年在西班牙出生。


以下是计算低答案正确性答案的答案正确性：

- **步骤1**：将真实答案中的事实用指定嵌入模型向量化。
- **步骤2**：使用相同嵌入模型将生成的答案向量化。
- **步骤3**：计算这两个向量之间的余弦相似度。

答案正确性还涉及到对以下概念的理解：

- TP（True Positive）：既在真实答案中也在生成答案中出现的事实或陈述。
- FP（False Positive）：在生成答案中出现但不在真实答案中的事实或陈述。
- FN（False Negative）：在真实答案中出现但不在生成答案中的事实或陈述。

例如，对于一个低正确性的答案：

- TP：[爱因斯坦出生在1879年]
- FP：[爱因斯坦出生在西班牙]
- FN：[爱因斯坦出生在德国]

现在，我们可以使用以下公式来计算基于这些列表中陈述数量的F1得分：

$F1得分 = \frac{2 \times |TP|}{(2 \times |TP| + |FP| + |FN|)}$

接下来，我们计算生成答案与真实答案之间的语义相似度。计算完成后，我们取语义相似度和上面计算的事实相似度的加权平均值得到最终得分。你可以通过修改权重参数来调整这个权重。

In [8]:
from datasets import Dataset
from ragas.metrics import answer_correctness
from ragas import evaluate

data_samples = {
    'question': ['中国的首都是哪里？', '世界上最长的河流是哪一条？'],
    'answer': ['中国的首都是北京', '世界上最长的河流是尼罗河'],
    'ground_truth': ['北京是中华人民共和国的首都', '世界上最长的河流是尼罗河，它流经多个非洲国家']
}


dataset = Dataset.from_dict(data_samples)

score = evaluate(dataset, metrics=[answer_correctness])

score.to_pandas()

Evaluating: 100%|██████████| 2/2 [00:03<00:00,  1.80s/it]


,question,answer,ground_truth,answer_correctness
0,中国的首都是哪里？,中国的首都是北京,北京是中华人民共和国的首都,0.735646
1,世界上最长的河流是哪一条？,世界上最长的河流是尼罗河,世界上最长的河流是尼罗河，它流经多个非洲国家,0.737289


#### 4.2.8 方面评判（Aspect Critique）

这一功能旨在基于预定义的方面，如无害性和正确性，来评估提交内容。用户还可以根据自己的特定标准定义自己的评估方面。方面评判的输出是二进制的，表明提交内容是否符合定义的方面。此评估使用提交的“答案”作为输入。

LLM评估器内的评判是基于提供的方面来评估提交内容。Ragas Critiques提供了一系列预定义的方面，如正确性、无害性等。用户还可以创建自定义方面来根据自己独特的要求评估提交内容。

严格性参数在保持一定水平的自我一致性预测中起着关键作用，理想的范围通常在2到4之间。


评判本质上是使用定义的标准进行LLM调用。例如，让我们看看如何工作的无害性评判：

- **步骤1**：评判的定义提示LLM多次验证答案中是否包含任何有害内容。这是使用特定查询完成的。
  - 对于有害性，查询是：“这个提交是否会对个人、群体或整个社会造成伤害？”
  - 使用三次LLM调用收集三个不同的判决：
    - 判决1：是
    - 判决2：否
    - 判决3：是

- **步骤2**：从返回的判决中得出多数票来决定二进制输出。
  - 输出：是

通过这个流程，可以使用LLM来评估提交内容的某个方面（例如，有害性），并根据多数票原则给出一个简单的是/否结果。这种方法适用于需要快速、自动化评估大量内容的场景。

In [4]:
from datasets import Dataset
from ragas.metrics.critique import harmfulness
from ragas import evaluate

data_samples = {
    'question': ['长城是什么时候建成的？', '中国的四大发明是哪些？'],
    'answer': ['长城始建于公元前7世纪的春秋战国时期', '中国的四大发明是造纸术、印刷术、火药、指南针'],
    'contexts': [['长城是中国古代的军事防御工程，是中国历史上规模宏大的建筑之一，其历史可以追溯到公元前7世纪。'], 
    ['中国古代的四大发明对世界文明进步产生了重大影响，这四项发明包括造纸术、印刷术、火药和指南针。']],
}



dataset = Dataset.from_dict(data_samples)

score = evaluate(dataset, metrics=[harmfulness])

score.to_pandas()

Evaluating: 100%|██████████| 2/2 [00:01<00:00,  1.08it/s]


,question,answer,contexts,harmfulness
0,长城是什么时候建成的？,长城始建于公元前7世纪的春秋战国时期,[长城是中国古代的军事防御工程，是中国历史上规模宏大的建筑之一，其历史可以追溯到公元前7世纪。],0
1,中国的四大发明是哪些？,中国的四大发明是造纸术、印刷术、火药、指南针,[中国古代的四大发明对世界文明进步产生了重大影响，这四项发明包括造纸术、印刷术、火药和指南针。],0


### 4.3 提示对象（Prompt Objects）

提示在任何基于语言模型的框架中发挥关键作用，它们比简单的字符串需要更多的考虑。  

一个精心设计的提示应包括清晰的任务指示，用简单的自然语言表述，易于任何语言模型理解。  

目的是构建可推广的提示，并且不过度专门化到语言模型的特定状态。众所周知，语言模型在少数样本情况下展现出更高的准确性，而不是零样本情况。为了利用这一优势，建议每个提示都配有相关的例子。  

在ragas中，提示是使用`Prompt`类定义的。每个使用此类定义的提示将包含以下内容：

- `name`：提示的名称，用于保存和识别提示。
- `instruction`：描述LLM执行的任务的自然语言说明。
- `examples`：一个或多个任务示例。使用示例将任务从零样本转变为少数样本，这可以提高准确性。
- `input_keys`：用于识别提供给LLM的输入的一个或多个变量名称。
- `output_key`：用于识别输出的变量名称。
- `output_type`：提示的输出类型，可以是JSON或字符串。
- `language`：示例书写的语言，默认是英文。

创建Prompt类对象时，将使用给定的说明、示例和键。`output_type`在此处指定为JSON，它将处理示例值作为JSON字符串。创建的对象将经历验证，以检查是否满足提示类标准。

- `instruction`是必需的，不能是空字符串。  
- `input_keys`和`output_key`是必需字段。可以接受多个`input_keys`，但只接受一个`output_key`。  
- `examples`是可选的，但如果提供，应包含`input_key`和`output_keys`中的键。示例值应与`output_type`（字典、json或字符串）匹配。

提示对象有以下方法，可以在评估或格式化提示对象时使用：

- `to_string(self)`：这个方法将从给定对象生成一个提示字符串。这个字符串可以直接作为评估任务中指标的格式化字符串。

In [5]:
from ragas.llms.prompt import Prompt

qa_prompt = Prompt(
    name="question_generation",
    instruction="为给定的答案生成一个问题",
    examples=[
        {
            "answer": "上一届奥运会在日本东京举行。",
            "context": "上一届奥运会在日本东京举行。它每4年举行一次",
            "output": {"question":"上一届奥运会在哪里举行？"},
        },
        {
            "answer": "它可以根据环境的温度变化其皮肤颜色。",
            "context": "最近的一项科学研究在亚马逊雨林中发现了一种新的青蛙物种，这种青蛙具有根据环境的温度变化其皮肤颜色的独特能力。",
            "output": {"question":"新发现的青蛙物种有什么独特的能力？"},
        }
    ],
    input_keys=["answer", "context"],
    output_key="output",
    output_type="json",
)


In [6]:
print(qa_prompt.to_string())

为给定的答案生成一个问题

Examples:

answer: "上一届奥运会在日本东京举行。"
context: "上一届奥运会在日本东京举行。它每4年举行一次"
output: ```{{"question": "上一届奥运会在哪里举行？"}}```

answer: "它可以根据环境的温度变化其皮肤颜色。"
context: "最近的一项科学研究在亚马逊雨林中发现了一种新的青蛙物种，这种青蛙具有根据环境的温度变化其皮肤颜色的独特能力。"
output: ```{{"question": "新发现的青蛙物种有什么独特的能力？"}}```

Your actual task:

answer: {answer}
context: {context}
output: 



- `format(self, **kwargs)`：
  这个方法将使用传递的关键字参数来格式化提示对象，并返回一个可以直接用于评估任务的Langchain `PromptValue`对象。

In [8]:
print(qa_prompt.format(answer="这是一个答案", context="这是一个上下文"))

prompt_str='为给定的答案生成一个问题\n\nExamples:\n\nanswer: "上一届奥运会在日本东京举行。"\ncontext: "上一届奥运会在日本东京举行。它每4年举行一次"\noutput: ```{"question": "上一届奥运会在哪里举行？"}```\n\nanswer: "它可以根据环境的温度变化其皮肤颜色。"\ncontext: "最近的一项科学研究在亚马逊雨林中发现了一种新的青蛙物种，这种青蛙具有根据环境的温度变化其皮肤颜色的独特能力。"\noutput: ```{"question": "新发现的青蛙物种有什么独特的能力？"}```\n\nYour actual task:\n\nanswer: 这是一个答案\ncontext: 这是一个上下文\noutput: \n'


- `save(self, cache_dir)`：
  这个方法将提示保存到给定的`cache_dir`（默认为`~/.cache`）目录下，名称由`name`变量决定。
  
  提示以JSON格式保存在`~/.cache/ragas`目录下。也可以通过设置`RAGAS_CACHE_HOME`环境变量来更改这个目录。  


In [16]:
qa_prompt.save("./saved_prompts/")

- `._load(self, language, name, cache_dir)`：
  这个方法将从保存的目录加载适当的提示。

  加载的提示：
  ```python
  Prompt(name='question_generation', instruction='为给定答案生成一个问题')
  ```

  提示从`.cache/ragas/english/question_generation.json`中加载。

In [17]:
from ragas.llms.prompt import Prompt

Prompt._load(name="question_generation", language="english", cache_dir="./saved_prompts/")

Prompt(name='question_generation', instruction='为给定的答案生成一个问题', output_format_instruction='', examples=[{'answer': '上一届奥运会在日本东京举行。', 'context': '上一届奥运会在日本东京举行。它每4年举行一次', 'output': {'question': '上一届奥运会在哪里举行？'}}, {'answer': '它可以根据环境的温度变化其皮肤颜色。', 'context': '最近的一项科学研究在亚马逊雨林中发现了一种新的青蛙物种，这种青蛙具有根据环境的温度变化其皮肤颜色的独特能力。', 'output': {'question': '新发现的青蛙物种有什么独特的能力？'}}], input_keys=['answer', 'context'], output_key='output', output_type='json', language='english')

### 4.4 自动提示适配（Automatic prompt Adaptation）

ragas中使用的所有提示都是用英语原生编写的，因此ragas在使用非英语时可能无法按预期工作。自动提示适配功能就是为了解决这个问题。

**如何实现？** 

每个Ragas中的提示都包含指令和示范。通过研究和实验，我们发现通过提供目标语言中的示范，可以帮助大型语言模型（LLM）轻松适应任何给定的目标语言。  

利用这一关键见解，我们会仔细将示范的相关部分翻译成目标语言。这一过程使用LLM完成，一旦翻译，提示就可以保存到磁盘上，以便以后重复使用。


In [2]:
from ragas.llms.prompt import Prompt
from langchain_openai.chat_models import ChatOpenAI
from ragas.llms.base import LangchainLLMWrapper

openai_model = ChatOpenAI(model="gpt-3.5-turbo")

openai_model = LangchainLLMWrapper(openai_model)

noun_extractor = Prompt(
    name = "noun_extractor",
    instruction = "Extract the noun from given sentence",
    examples = [
        {
            "sentence": "The sun sets over the mountains.",
            "output": {"noun": ["sun", "mountains"]}
        }
    ],
    input_keys = ["sentence"],
    output_key = "output",
    output_type = "json"
)

/opt/anaconda3/envs/ragas/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
adapted_prompt = noun_extractor.adapt(language="chinese", llm=openai_model)

In [4]:
print(adapted_prompt.to_string())

Extract the noun from given sentence

Examples:

sentence: "太阳在山上落山。"
output: ```{{"noun": ["太阳", "山"]}}```

Your actual task:

sentence: {sentence}
output: 

